In [1]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# ---------- Config ----------
INPUT_PATH = "hn_scored_latest.csv"   # from your daily pipeline
INDEX_PATH = "hn_rag.index"
META_PATH = "hn_rag_meta.pkl"

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_COL = "clean_text"
THEME_COL = "market_theme"

# ---------- Load ----------
df = pd.read_csv(INPUT_PATH)
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
df = df[df[TEXT_COL].str.len() >= 40].copy()

# ---------- Build docs ----------
docs = []
meta = []

for _, r in df.iterrows():
    text = r.get(TEXT_COL, "")
    title = str(r.get("story_title", "") or "")
    theme = str(r.get(THEME_COL, "") or "")
    url = str(r.get("url", "") or "")
    created_at = str(r.get("created_at", "") or "")
    keyword = str(r.get("keyword", "") or "")
    objectID = str(r.get("objectID", "") or "")

    # A little formatting helps retrieval quality
    doc = f"Title: {title}\nTheme: {theme}\nKeyword: {keyword}\nDate: {created_at}\nText: {text}\nURL: {url}"

    docs.append(doc)
    meta.append({
        "story_title": title,
        "market_theme": theme,
        "keyword": keyword,
        "created_at": created_at,
        "url": url,
        "objectID": objectID
    })

print("Docs:", len(docs))

# ---------- Embed ----------
embedder = SentenceTransformer(MODEL_NAME)
emb = embedder.encode(docs, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True).astype("float32")

# ---------- FAISS index ----------
d = emb.shape[1]
index = faiss.IndexFlatIP(d)  # cosine similarity if vectors are normalized
index.add(emb)

faiss.write_index(index, INDEX_PATH)
with open(META_PATH, "wb") as f:
    pickle.dump({"meta": meta, "docs": docs}, f)

print("Saved:", INDEX_PATH, META_PATH)

/opt/anaconda3/envs/shid_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/shid_env/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /opt/anaconda3/envs/shid_env/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


Docs: 1003


Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1426.63it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 32/32 [00:02<00:00, 13.10it/s]

Saved: hn_rag.index hn_rag_meta.pkl


In [1]:
import torch, transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

/opt/anaconda3/envs/shid_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.10.0
transformers: 5.1.0


In [2]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

INDEX_PATH = "hn_rag.index"
META_PATH = "hn_rag_meta.pkl"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K = 8

# Load
index = faiss.read_index(INDEX_PATH)
with open(META_PATH, "rb") as f:
    store = pickle.load(f)

docs = store["docs"]
meta = store["meta"]

embedder = SentenceTransformer(MODEL_NAME)

def search(query: str, top_k: int = TOP_K, theme_filter: str | None = None):
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, top_k * 5)  # over-retrieve then filter

    results = []
    for sc, i in zip(scores[0], idx[0]):
        if i == -1:
            continue
        m = meta[i]
        if theme_filter and m.get("market_theme") != theme_filter:
            continue
        results.append((float(sc), m, docs[i]))
        if len(results) >= top_k:
            break
    return results

if __name__ == "__main__":
    q = input("Ask: ").strip()
    results = search(q, top_k=8)

    for rank, (sc, m, doc) in enumerate(results, start=1):
        print("\n" + "="*80)
        print(f"{rank}) score={sc:.3f} | theme={m.get('market_theme')} | keyword={m.get('keyword')}")
        print(f"   title: {m.get('story_title')}")
        print(f"   url:   {m.get('url')}")
        print("-"*80)
        # show just the “Text:” part
        text_part = doc.split("\nText:", 1)[-1].strip()
        print(text_part[:700], "..." if len(text_part) > 700 else "")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1647.94it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ask:  quit



1) score=0.247 | theme=Developer Ecosystem & Integrations | keyword=creator
   title: Ask HN: What are you working on? (February 2026)
   url:   https://news.ycombinator.com/item?id=46952960
--------------------------------------------------------------------------------
Well. I created an account called 'cfltz' and started posting summarization of news items that were discussed on the subreddit. I really have an ideological 'war' with how news is presented and I felt like it was really a good product for expanding the discourse. Then after I got enough Karma, I think 500, after two months (I didn't really post religiously, just things that had my interest) I posted a link to the actual website instead of just a summarization. Within 60 seconds I got banned from the subreddit. I contacted the mods and asked what's up and they told me self-promotion is not allowed. So that was that. I accepted the resolution, deleted the account and got back to my main, which ...

2) score=0.227 | them

In [ ]:
import faiss
import pickle
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
from sentence_transformers import SentenceTransformer

INDEX_PATH = "hn_rag.index"
META_PATH = "hn_rag_meta.pkl"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

TOP_K = 8

# Load
index = faiss.read_index(INDEX_PATH)
with open(META_PATH, "rb") as f:
    store = pickle.load(f)

docs = store["docs"]
meta = store["meta"]

embedder = SentenceTransformer(MODEL_NAME)

# ---- Filters ----
PAIN_CUES = [
    "pain", "pain point", "problem", "issue", "hard", "difficult", "friction",
    "annoy", "struggle", "fails", "confusing", "complain", "pitfall",
    "refund", "refunds", "chargeback", "chargebacks", "compliance",
    "access control", "entitlement", "entitlements", "permissions", "oauth"
]

def has_pain_language(text: str) -> bool:
    t = str(text).lower()
    return any(w in t for w in PAIN_CUES)

def is_within_days(created_at, days=7) -> bool:
    """
    Returns True if created_at is within the last `days` days.
    Works with ISO timestamps and most parseable formats.
    """
    dt = pd.to_datetime(created_at, utc=True, errors="coerce")
    if pd.isna(dt):
        return False
    return dt >= (datetime.now(timezone.utc) - timedelta(days=days))

def search(
    query: str,
    top_k: int = TOP_K,
    theme_filter: str | None = None,
    days: int | None = 7,
    require_pain: bool = True
):
    q = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")

    # Over-retrieve so filtering doesn't starve results
    overfetch = max(top_k * 50, 200)
    scores, idx = index.search(q, overfetch)

    results = []
    for sc, i in zip(scores[0], idx[0]):
        if i == -1:
            continue

        m = meta[i]
        doc = docs[i]

        # ✅ This-week filter
        if days is not None and not is_within_days(m.get("created_at", ""), days=days):
            continue

        # ✅ Optional theme filter
        if theme_filter and m.get("market_theme") != theme_filter:
            continue

        # ✅ Pain-aware filter
        if require_pain and not has_pain_language(doc):
            continue

        results.append((float(sc), m, doc))
        if len(results) >= top_k:
            break

    return results

if __name__ == "__main__":
    print("RAG ready. Type a question (or 'exit' to quit).\n")

    while True:
        q = input("Ask: ").strip()
        if q.lower() in {"exit", "quit", "q"}:
            print("Goodbye 👋")
            break

        results = search(
            q,
            top_k=8,
            days=7,
            require_pain=True
        )

        if not results:
            print("\nNo relevant results found for this week.\n")
            continue

        for rank, (sc, m, doc) in enumerate(results, start=1):
            print("\n" + "="*80)
            print(f"{rank}) score={sc:.3f} | theme={m.get('market_theme')} | keyword={m.get('keyword')}")
            print(f"   title: {m.get('story_title')}")
            print(f"   url:   {m.get('url')}")
            print("-"*80)
            text_part = doc.split("\nText:", 1)[-1].strip()
            print(text_part[:700], "..." if len(text_part) > 700 else "")

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 1457.99it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RAG ready. Type a question (or 'exit' to quit).



Ask:  Whats new this week?



No relevant results found for this week.



Ask:  what is complaints are there regarding film studios and funding?



No relevant results found for this week.



Ask:  monetisation



No relevant results found for this week.

